In [2]:
import tensorflow as tf
import numpy as np
import xml.etree.ElementTree as ET
import os

In [3]:
def parse_voc_annotations(annotation_dir, image_dir, class_map):
    image_paths = []
    bbox_data = []
    class_labels =[]
    for xml_file in sorted(os.listdir(os.path.join(annotation_dir))):
        if not xml_file.endswith(".xml"):
            continue
        tree = ET.parse(os.path.join(annotation_dir, xml_file))
        root = tree.getroot()
        image_file_name = root.find("filename").text
        path = os.path.join(image_dir,image_file_name)

        size = root.find("size")
        img_width = int(size.find("width").text)
        img_height = int(size.find("height").text)

        obj = root.find("object")
        if obj is not None:
            class_name = obj.find("name").text
            if class_name not in class_map:
                continue
            class_id = class_map[class_name]

            bndbox = obj.find("bndbox")
            xmin = float(bndbox.find("xmin").text) / img_width
            ymin = float(bndbox.find("ymin").text) / img_height
            xmax = float(bndbox.find("xmax").text) / img_width
            ymax = float(bndbox.find("ymax").text) / img_height
            image_paths.append(path)
            bbox_data.append([xmin, ymin, xmax, ymax])
            class_labels.append(class_id)
    return image_paths, bbox_data, class_labels



In [4]:
image_directory = "../data/images"
annotations_directory = "../data/Annotations"
class_map ={"thankyou":0,"hello":1,"okay":3}
image_paths, bounding_box,class_labels= parse_voc_annotations(annotations_directory,image_directory,class_map)

In [ ]:
image_paths = tf.constant(image_paths)
bbox_data = tf.constant(bounding_box, dtype=tf.float32)
class_labels = tf.constant(class_labels, dtype=tf.int32)

def load_and_preprocess_image(path, bbox, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [128, 128])
    image = image / 255.0
    label_one_hot = tf.one_hot(label, depth=len(class_map))
    return image, {"gesture": label_one_hot, "bbox": bbox}

In [15]:
raw_ds = tf.data.Dataset.from_tensor_slices((image_paths, bbox_data, class_labels))

DATASET_SIZE = len(image_paths)
train_size  = int(0.8 * DATASET_SIZE)
raw_ds      = raw_ds.shuffle(buffer_size=DATASET_SIZE)

train_ds = raw_ds.take(train_size)
val_ds   = raw_ds.skip(train_size)

train_ds = train_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds   = val_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.batch(32).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.batch(32).prefetch(tf.data.AUTOTUNE)


In [36]:
def augment(image, targets):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.2)
    image = tf.image.random_contrast(image, 0.5, 1.5)
    return image, targets

train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)


In [38]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D,BatchNormalization,Dropout
from tensorflow.keras.models import Model


base_model = MobileNetV2(input_shape=(128,128,3), include_top=False, weights="imagenet")

base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False
for layer in base_model.layers[-50:]:
    layer.trainable = True


In [46]:
inputs = Input(shape=(128,128,3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

bbox = Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(1e-4))(x)
bbox = Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(1e-4))(bbox)
bbox = Dense(4, name='bbox', activation='sigmoid')(bbox)  

cls = Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(1e-4))(x)
cls = Dense(len(class_map), name='gesture', activation='softmax')(cls)

model = Model(inputs=inputs, outputs=[bbox, cls])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss={
      'bbox': tf.keras.losses.mae,         
      'gesture': 'categorical_crossentropy'
    },
    loss_weights={
    'gesture': 1.5,
    'bbox': 0.5,        
  },
    metrics={'gesture': 'accuracy'}
)

In [27]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_1… │ (None, 4, 4,      │  2,257,984 │ input_layer_8[0]… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1280)      │      5,120 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 1280)      │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │    163,968 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │      8,256 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │     81,984 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bbox (Dense)        │ (None, 4)         │        260 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gesture (Dense)     │ (None, 3)         │        195 │ dense_5[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,517,767 (9.60 MB)

 Trainable params: 1,463,303 (5.58 MB)

 Non-trainable params: 1,054,464 (4.02 MB)

In [47]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[
      tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5),
      tf.keras.callbacks.EarlyStopping(
          monitor="val_gesture_accuracy",
            patience=10,
            restore_best_weights=True,
            mode="max",
            verbose=1
      ),
      tf.keras.callbacks.ModelCheckpoint(
          os.path.join("../",'trained model/',"best_sign_detector.keras"),
            monitor="val_gesture_accuracy",
            save_best_only=True,
            save_weights_only=False,
            mode="max",          
            verbose=1
      )
    ],
    verbose=1

)

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 482ms/step - bbox_loss: 0.2769 - gesture_accuracy: 0.4308 - gesture_loss: 0.8821 - loss: 1.4788
Epoch 1: val_gesture_accuracy improved from -inf to 0.21429, saving model to ../trained model/best_sign_detector.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 47s 8s/step - bbox_loss: 0.2812 - gesture_accuracy: 0.4182 - gesture_loss: 0.9222 - loss: 1.5323 - val_bbox_loss: 0.2899 - val_gesture_accuracy: 0.2143 - val_gesture_loss: 0.4069 - val_loss: 0.7993 - learning_rate: 1.0000e-05
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 534ms/step - bbox_loss: 0.2886 - gesture_accuracy: 0.3750 - gesture_loss: 0.6030 - loss: 1.0876
Epoch 2: val_gesture_accuracy improved from 0.21429 to 0.35714, saving model to ../trained model/best_sign_detector.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 3s/step - bbox_loss: 0.2924 - gesture_accuracy: 0.3750 - gesture_loss: 0.6098 - loss: 1.0979 - val_bbox_loss: 0.3168 - val_gesture_accuracy: 0.3571 - val_gesture_loss: 0.2995 - val_loss: 0.6517 - learning_rate:

In [50]:
import cv2

# 1️⃣ Load your saved model
model = tf.keras.models.load_model(os.path.join("../","trained model/best_sign_detector.keras"), compile=False)

# 2️⃣ Helper to run one forward pass
def detect_frame(frame):
    img = cv2.resize(frame, (128,128))
    img = img.astype('float32') / 255.0
    inp = np.expand_dims(img, 0)
    bbox_pred, cls_pred = model.predict(inp)
    # undo normalization:
    h, w = frame.shape[:2]
    xmin, ymin, xmax, ymax = bbox_pred[0]
    return (
      int(xmin*w), int(ymin*h),
      int((xmax-xmin)*w), int((ymax-ymin)*h)
    ), np.argmax(cls_pred[0])

# 3️⃣ Initialize video & tracker list
cap = cv2.VideoCapture(0)
trackers = []
FRAME_DETECT_INTERVAL = 10

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 🕵️‍♂️ Every N frames: re‑detect
    if frame_idx % FRAME_DETECT_INTERVAL == 0 or len(trackers)==0:
        trackers = []  # reset
        box, cls_id = detect_frame(frame)
        # create a new CSRT tracker
        tracker = cv2.TrackerCSRT_create()
        tracker.init(frame, box)
        trackers.append((tracker, cls_id))

    else:
        # just update existing trackers
        for tracker, cls_id in trackers:
            ok, box = tracker.update(frame)
            if not ok:
                continue
            x,y,w,h = map(int, box)
            # draw
            cv2.rectangle(frame, (x,y), (x+w, y+h), (0,255,0), 2)
            label = list(class_map.keys())[list(class_map.values()).index(cls_id)]
            cv2.putText(frame, label, (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

    cv2.imshow("Sign Language Detector + Tracker", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    frame_idx += 1

cap.release()
cv2.destroyAllWindows()


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


AttributeError: module 'cv2' has no attribute 'TrackerCSRT_create'

In [58]:
import cv2
# import numpy as np
# import tensorflow as tf

# 1️⃣ Load your saved model
model = tf.keras.models.load_model(os.path.join("../","trained model/best_sign_detector.keras"), compile=False)

# 2️⃣ Helper to run one forward pass
def detect_frame(frame):
    img = cv2.resize(frame, (128,128))
    img = img.astype('float32') / 255.0
    inp = np.expand_dims(img, 0)
    bbox_pred, cls_pred = model.predict(inp, verbose=0)
    h, w = frame.shape[:2]
    xmin, ymin, xmax, ymax = bbox_pred[0]
    # convert normalized to pixel coords
    x, y = int(xmin*w), int(ymin*h)
    bw, bh = int((xmax-xmin)*w), int((ymax-ymin)*h)
    return (x, y, bw, bh), np.argmax(cls_pred[0])

# 3️⃣ Initialize video & CamShift state
cap = cv2.VideoCapture(0)
FRAME_DETECT_INTERVAL = 10
frame_idx = 0

track_window = None
roi_hist = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 🕵️‍♂️ Re-detect every N frames or if no valid window
    if frame_idx % FRAME_DETECT_INTERVAL == 0 or track_window is None:
        box, cls_id = detect_frame(frame)
        x, y, w, h = box
        track_window = (x, y, w, h)

        # --- Initialize ROI histogram for CamShift ---
        roi = frame[y:y+h, x:x+w]
        # hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        # mask out low/light areas
        # mask = cv2.inRange(hsv_roi, np.array((0., 30.,32.)), np.array((180.,255.,255.)))
        # roi_hist = cv2.calcHist([hsv_roi], [0], mask, [180], [0,180])
        # cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

        # draw initial detection
        label = list(class_map.keys())[list(class_map.values()).index(cls_id)]
        cv2.rectangle(frame, (x,y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, label, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

    else:
        pass
        # # --- CamShift tracking step ---
        # hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        # back_proj = cv2.calcBackProject([hsv], [0], roi_hist, [0,180], 1)
        # # apply CamShift to get new location
        # ret, track_window = cv2.CamShift(back_proj, track_window,
        #                                  (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1))
        # pts = cv2.boxPoints(ret)
        # pts = np.int0(pts)
        # # Draw rotated bounding box
        # cv2.polylines(frame, [pts], True, (0,255,0), 2)

    cv2.imshow("Sign Language Detector + CamShift Tracker", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    frame_idx += 1

cap.release()
cv2.destroyAllWindows()
